# Análisis OLAP - Data Warehouse Gold

## Objetivo
Realizar análisis multidimensional sobre el Data Warehouse ya cargado

## Dataset
- **Fuente:** Data Warehouse (DuckDB o PostgreSQL)
- **Tablas:** 7 (6 dimensiones + 1 hechos)
- **Registros:** ~500,000 operaciones

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## Conexión a Data Warehouse

In [ ]:
# Conectar a DuckDB
conn = duckdb.connect('data_lake/gold/aduana.duckdb')

# Verificar tablas
tablas = conn.execute("SELECT * FROM information_schema.tables").fetchall()
print("Tablas en el Data Warehouse:")
for tabla in tablas:
    print(f"  - {tabla[2]}")

## Análisis Temporal

In [ ]:
# Evolución mensual
query = """
SELECT 
    df.año,
    df.mes,
    df.nombre_mes,
    COUNT(DISTINCT f.fact_item_id) as operaciones,
    SUM(f.valor_cif) as valor_cif_total
FROM fact_aduana_item f
JOIN dim_fecha df ON f.fecha_id = df.fecha_id
GROUP BY df.año, df.mes, df.nombre_mes
ORDER BY df.año, df.mes
"""

df_temporal = conn.execute(query).fetchdf()
print(df_temporal)

In [ ]:
# Visualizar evolución temporal
plt.figure(figsize=(14, 6))
for year in df_temporal['año'].unique():
    data = df_temporal[df_temporal['año'] == year]
    plt.plot(data['mes'], data['valor_cif_total']/1e9, marker='o', label=str(year))

plt.xlabel('Mes')
plt.ylabel('Valor CIF (Billones USD)')
plt.title('Evolución Mensual de Importaciones')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Análisis Geográfico

In [ ]:
# Top 10 países
query = """
SELECT 
    dp.nombre_pais,
    COUNT(DISTINCT f.fact_item_id) as operaciones,
    SUM(f.valor_cif) as valor_cif,
    ROUND(SUM(f.valor_cif) * 100.0 / (SELECT SUM(valor_cif) FROM fact_aduana_item), 2) as pct_total
FROM fact_aduana_item f
JOIN dim_pais dp ON f.pais_id = dp.pais_id
GROUP BY dp.nombre_pais
ORDER BY valor_cif DESC
LIMIT 10
"""

df_paises = conn.execute(query).fetchdf()
print(df_paises)

In [ ]:
# Visualizar top países
plt.figure(figsize=(12, 6))
plt.barh(df_paises['nombre_pais'], df_paises['valor_cif']/1e9, color='skyblue')
plt.xlabel('Valor CIF (Billones USD)')
plt.title('Top 10 Países Importadores')
plt.tight_layout()
plt.show()

## Análisis por Producto

In [ ]:
# Top 15 productos
query = """
SELECT 
    dp.descripcion_item,
    dp.capitulo,
    COUNT(DISTINCT f.fact_item_id) as operaciones,
    SUM(f.valor_cif) as valor_cif
FROM fact_aduana_item f
JOIN dim_producto dp ON f.producto_id = dp.producto_id
GROUP BY dp.descripcion_item, dp.capitulo
ORDER BY valor_cif DESC
LIMIT 15
"""

df_productos = conn.execute(query).fetchdf()
print(df_productos)

In [ ]:
# Análisis por capítulo
query = """
SELECT 
    dp.capitulo,
    COUNT(DISTINCT dp.producto_id) as cantidad_productos,
    SUM(f.valor_cif) as valor_total
FROM fact_aduana_item f
JOIN dim_producto dp ON f.producto_id = dp.producto_id
GROUP BY dp.capitulo
ORDER BY valor_total DESC
LIMIT 10
"""

df_capitulos = conn.execute(query).fetchdf()
print(df_capitulos)

## Análisis Operativo

In [ ]:
# Volumen por aduana
query = """
SELECT 
    da.nombre_aduana,
    COUNT(DISTINCT f.fact_item_id) as operaciones,
    SUM(f.valor_cif) as valor_cif,
    ROUND(SUM(f.valor_cif) * 100.0 / (SELECT SUM(valor_cif) FROM fact_aduana_item), 2) as pct_total
FROM fact_aduana_item f
JOIN dim_aduana da ON f.aduana_id = da.aduana_id
GROUP BY da.nombre_aduana
ORDER BY valor_cif DESC
"""

df_aduanas = conn.execute(query).fetchdf()
print(df_aduanas)

In [ ]:
# Visualizar aduanas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico de barras
ax1.bar(df_aduanas['nombre_aduana'], df_aduanas['operaciones'], color='coral')
ax1.set_title('Operaciones por Aduana')
ax1.set_ylabel('Cantidad')
ax1.tick_params(axis='x', rotation=45)

# Gráfico de pastel
ax2.pie(df_aduanas['pct_total'], labels=df_aduanas['nombre_aduana'], autopct='%1.1f%%')
ax2.set_title('Participación de Aduanas')

plt.tight_layout()
plt.show()

## Análisis CIF vs FOB

In [ ]:
# Análisis global CIF vs FOB
query = """
SELECT 
    COUNT(*) as operaciones,
    ROUND(AVG(valor_cif), 2) as promedio_cif,
    ROUND(AVG(valor_fob), 2) as promedio_fob,
    ROUND(SUM(valor_cif), 2) as suma_cif,
    ROUND(SUM(valor_fob), 2) as suma_fob,
    ROUND(AVG(valor_cif - valor_fob), 2) as margen_promedio
FROM fact_aduana_item
"""

df_cif_fob = conn.execute(query).fetchdf()
print(df_cif_fob)

## Matriz de Análisis Cruzado

In [ ]:
# Matriz: País × Operación
query = """
SELECT 
    dp.nombre_pais,
    do.nombre_operacion,
    COUNT(*) as operaciones,
    ROUND(SUM(f.valor_cif), 2) as valor_cif
FROM fact_aduana_item f
JOIN dim_pais dp ON f.pais_id = dp.pais_id
JOIN dim_operacion do ON f.operacion_id = do.operacion_id
GROUP BY dp.nombre_pais, do.nombre_operacion
ORDER BY valor_cif DESC
LIMIT 20
"""

df_cruzado = conn.execute(query).fetchdf()
print(df_cruzado)

## Conclusiones del Análisis OLAP

- Data Warehouse está operacional
- Todas las dimensiones están cargadas
- Análisis multidimensionales disponibles
- Datos listos para Power BI